# Linear Regression, Explained Simply
Fitting the best straight line through data, by hand and with scikit-learn. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A food delivery app wants to predict how long a delivery takes from the distance. We have 80 past deliveries.

### 80 deliveries
- `deliveries.csv`: 80 deliveries: the distance in kilometres, and how many minutes it took
- `df.head()`: the first five deliveries

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "linear-regression/deliveries.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### Distance against time
- `plt.scatter`: one dot per delivery
- `the pattern`: the dots rise in a fairly straight band

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["distance_km"], df["minutes"])
plt.xlabel("distance (km)")
plt.ylabel("delivery time (minutes)")
plt.title("Longer trips take longer")
plt.show()

## 2. The best line, by hand
A straight line is **y = m x + b**: **b** is the intercept (where the line starts when x is 0) and **m** is the slope (how much y goes up for each 1 in x). The best line has the smallest **mean squared error**: the average of the squared vertical gaps between the dots and the line.

### Three points, by hand
- `x - x.mean()`: how far each x is from the average x
- `top: sum of products`: how x and y move together
- `bottom: sum of squares`: how much x moves on its own
- `b = y.mean() - m * x.mean()`: the best line always passes through the average point

In [ ]:
import numpy as np

x = np.array([1, 2, 3])
y = np.array([14, 19, 21])
top = ((x - x.mean()) * (y - y.mean())).sum()
bottom = ((x - x.mean()) ** 2).sum()
m = top / bottom
b = y.mean() - m * x.mean()
print("slope:", m, " intercept:", b)

### The same formula on all 80 deliveries
- `slope`: about 3.3 extra minutes for every extra kilometre
- `intercept`: about 12 minutes even for a tiny distance: the time to cook and pack the food

In [ ]:
x = df["distance_km"]
y = df["minutes"]
top = ((x - x.mean()) * (y - y.mean())).sum()
bottom = ((x - x.mean()) ** 2).sum()
m = top / bottom
b = y.mean() - m * x.mean()
print("slope:", round(m, 2), " intercept:", round(b, 2))

## 3. With scikit-learn
scikit-learn finds the same line in two lines of code.

### LinearRegression
- `df[["distance_km"]]`: double brackets: a table with one column, because scikit-learn expects a table of features
- `model.fit(X, y)`: find the line with the smallest mean squared error
- `coef_ and intercept_`: the slope and intercept it found: the same as our formula

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[["distance_km"]]
model = LinearRegression()
model.fit(X, y)
print("slope:", round(model.coef_[0], 2))
print("intercept:", round(model.intercept_, 2))

### Drawing the line
- `model.predict(X)`: the line's prediction for every delivery
- `plt.plot(..., color="red")`: join those predictions: the red line

In [ ]:
plt.scatter(df["distance_km"], df["minutes"], alpha=0.6)
plt.plot(df["distance_km"], model.predict(X), color="red")
plt.xlabel("distance (km)")
plt.ylabel("delivery time (minutes)")
plt.title("The best straight line")
plt.show()

### Predicting new deliveries
- `new`: three new distances, as a table with the same column name
- `zip`: pairs each distance with its prediction

In [ ]:
new = pd.DataFrame({"distance_km": [2, 5, 8]})
for km, mins in zip(new["distance_km"], model.predict(new)):
    print(km, "km ->", round(mins), "minutes")

## 4. How good is the line?
**MSE** is the mean squared error; its square root (**RMSE**) is in minutes. **R squared** compares the line with always guessing the average: 1 is perfect, 0 is no better than the average.

### Error and R squared
- `mean_squared_error`: the average squared gap between the real times and the line
- `mse ** 0.5`: the square root, back in minutes: a typical miss
- `r2_score`: 1 minus (the line's MSE divided by the MSE of always guessing the average)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = model.predict(X)
mse = mean_squared_error(y, pred)
print("MSE: ", round(mse, 1))
print("RMSE:", round(mse ** 0.5, 1), "minutes")
print("R squared:", round(r2_score(y, pred), 3))

### A fair test on unseen deliveries
- `train_test_split`: keep 25% of deliveries hidden while the line is fitted
- `test RMSE`: the typical miss on deliveries the model never saw

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0)
model2 = LinearRegression().fit(X_train, y_train)
test_mse = mean_squared_error(y_test, model2.predict(X_test))
print("test RMSE:", round(test_mse ** 0.5, 1))

## 5. Watch out
A straight line can be pulled by unusual points, and it should not be trusted far outside the data.

### One traffic jam
- `jam`: one strange delivery: 1 km that took 75 minutes
- `pd.concat`: add it to the data
- `before / after`: one point changes the slope and intercept noticeably

In [ ]:
jam = pd.DataFrame({"distance_km": [1.0], "minutes": [75]})
df2 = pd.concat([df, jam], ignore_index=True)
m2 = LinearRegression()
m2.fit(df2[["distance_km"]], df2["minutes"])
print("before:", round(model.coef_[0], 2), round(model.intercept_, 1))
print("after: ", round(m2.coef_[0], 2), round(m2.intercept_, 1))

### Predicting far outside the data
- `100 km`: our data only goes up to 10 km
- `the answer`: the line happily answers, but nothing in the data says it holds that far

In [ ]:
far = pd.DataFrame({"distance_km": [100]})
print(round(model.predict(far)[0]), "minutes for 100 km?")

## 6. Mini project: the delivery estimate
Turn the model into the message an app would show.

### An app message
- `estimate(km)`: predict the minutes for one distance
- `plus or minus rmse`: a simple range, using the typical miss
- `f-string`: build the message the customer sees

In [ ]:
def estimate(km):
    mins = model.predict(pd.DataFrame({"distance_km": [km]}))[0]
    rmse = mse ** 0.5
    low, high = round(mins - rmse), round(mins + rmse)
    return f"{km} km: about {round(mins)} min ({low} to {high})"

for km in [1.5, 4, 9]:
    print(estimate(km))

## Try it yourself
1. Predict the delivery time for 7.5 km.
2. Remove the traffic-jam delivery and fit again: how much do the slope and intercept change?
3. Work out the slope by hand for the points (1, 10), (2, 14), (3, 15).